# BTS Marketing Carrier On-Time Performance between 2018 and 2025

### Require libraries
The section below imports the required modules for the environment.

In [1]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import plotly.express as px
import pandas as pd
import warnings
from plotly.subplots import make_subplots
from ipyleaflet import Map, Polyline, CircleMarker, LayerGroup, basemaps, Marker
import plotly.graph_objects as go
warnings.filterwarnings("ignore")


### Utilities
The section below contains reusable utility functions used throughout this notebook.

In [1]:
def create_flow_plot(df, level1, level2, value):
    # build node list
    labels = pd.Index(pd.concat([df[level1], df[level2]]).unique())
    label_to_id = {label: i for i, label in enumerate(labels)}
    # map source/target ids
    df["source"] = df[level1].map(label_to_id)
    df["target"] =df[level2].map(label_to_id)
    #costume colors
    reason_colors = {
        "Carrier": "rgba(31,119,180,0.45)",
        "Weather": "rgba(255,127,14,0.45)",
        "NAS": "rgba(44,160,44,0.45)",
        "Security": "rgba(214,39,40,0.45)",
        "LateAircraft": "rgba(148,103,189,0.45)"}
    df["link_color"] = df[level2].map(reason_colors)
    fig = go.Figure(data=[
        go.Sankey(node=dict(pad=15,thickness=35,label=labels.tolist()),
                link=dict(source=df["source"],target=df["target"],
                 value=df[value],color=df["link_color"]))])
    fig.update_layout(font_size=12)
    return fig
def create_pie_plot(df, name_field, value_field, title=""):
    fig=px.pie(df, names=name_field, values=value_field, title=title)
    fig.update_traces(textinfo="percent+label", textposition="inside")
    return fig
def create_box_plot(df, x_field, y_field, color_field):
    fig= px.box(df,  x=x_field,  y=y_field, color=color_field,
                 points=False,log_y=True)
    fig.update_traces(line_width=0)
    return fig

def make_ipyleaflet_flight_map(plot_df ,min_count=0):
    plot_df = plot_df.dropna(subset=[
        "org_lat", "org_long", "dest_lat", "dest_long", "flight_count" ])
    plot_df = plot_df.groupby(["OriginStateName",  'DestStateName'],as_index=False
        ).agg(org_lat=("org_lat", "first"),org_long=("org_long", "first"),
        dest_lat=("dest_lat", "first"),dest_long=("dest_long", "first"),
        flight_count=("flight_count", "sum"))
    plot_df = plot_df[plot_df["flight_count"] >= min_count]
    center_lat = pd.concat([plot_df["org_lat"], plot_df["dest_lat"]]).mean()
    center_lon = pd.concat([plot_df["org_long"], plot_df["dest_long"]]).mean()
    m = Map(center=(center_lat, center_lon), zoom=5,basemap=basemaps.Esri.WorldStreetMap)
    max_count = plot_df["flight_count"].max()
    layers = []
    for _, row in plot_df.iterrows():
        weight =int(1 +15* (row["flight_count"] / max_count))
        layers.append(Polyline(locations=[
                    (row["org_lat"], row["org_long"]),
                    (row["dest_lat"], row["dest_long"])],
                weight=weight, color="navy"))
        layers.append(CircleMarker(location=(row["org_lat"], row["org_long"]), radius=2, color="green"))
        layers.append(CircleMarker(location=(row["dest_lat"], row["dest_long"]),radius=2, color="red"))
    m.add(LayerGroup(layers=layers))
    return m

months= ["January", "February", "March", "April", "May", "June",
        "July", "August", "September", "October", "November", "December"]


## list of fields that can be explored in the dashboard
explore_fields=['flight_count','DepTime','DepDelay','DepDelayMinutes','DepDel15','ArrTime',
 'ArrDelay','ArrDelayMinutes','ArrDel15','Cancelled','Diverted','AirTime',
 'Distance','CarrierDelay','WeatherDelay','NASDelay','SecurityDelay','LateAircraftDelay','FirstDepTime']




## Reading Input Dataset
The dataset used in this notebook was downloaded from the U.S. Department of Transportation website. You can access the dataset using the link below:
[Download link](https://www.transtats.bts.gov/DL_SelectFields.aspx?gnoyr_VQ=FGK&QO_fu146_anzr=)
The dataset spans the years 2018 to 2025 and contains more than 56 million records in total. For the purposes of this report, 2 million records were selected randomly using the pandas sample() function.

In [ ]:

## data url from github


main_data_path=r"https://github.com/Hasimengin/flight_dashboard/raw/main/On_Time_Marketing_Carrier_On_Time_Performance_update_sample_v2.parquet"
carriers_list_path=r"https://github.com/Hasimengin/flight_dashboard/raw/main/L_UNIQUE_CARRIERS.csv"
airport_locations_path=r"https://github.com/Hasimengin/flight_dashboard/raw/main/usa_airport_locations.csv"

# read data
main_df = pd.read_parquet(main_data_path, engine="fastparquet")
airport_loc=pd.read_csv(airport_locations_path)
carrier_list=pd.read_csv(carriers_list_path)

## recode cancellation reasons and merge airport location data
main_df['CancellationCode'] = main_df['CancellationCode']\
                            .str.replace({"A":	"Carrier","B":	"Weather",
                            "C":"National Air System","D":"Security"})
## add airport location data for both origin and destination airports
airport_loc_org=airport_loc.rename(  {'IATA':'Origin' , 'LATITUDE':"org_lat", 'LONGITUDE' :"org_long"}, axis=1)  
airport_loc_dest=airport_loc.rename(  {'IATA':'Dest' , 'LATITUDE':"dest_lat", 'LONGITUDE' :"dest_long"}, axis=1 )
main_df=main_df.merge(airport_loc_org[['Origin',"org_lat","org_long"]], on='Origin', how="left")
main_df=main_df.merge(airport_loc_dest[['Dest',"dest_lat","dest_long"]], on='Dest', how="left")

# attached carrier names
main_df=main_df.merge(carrier_list[['Code','Description']], left_on='Operating_Airline ', right_on='Code', how="left")
main_df=main_df.rename(columns={"Description":"Carrier_Name"})

## get extract year and month names attribute from data fiel
main_df["year"] =main_df['FlightDate'].dt.year
main_df["month"] =main_df['FlightDate'].dt.month_name()
main_df["quarter"] = main_df['FlightDate'].dt.quarter
main_df["quarter"] =main_df["quarter"].map({1: "Q1", 2: "Q2", 3: "Q3", 4: "Q4"})
main_df["flight_count"]=1
main_df["airport_name"]=main_df["Origin"].astype(str)+"_"+main_df['OriginCityName'].astype(str)

main_df["month"] = pd.Categorical(main_df["month"],categories=months,ordered=True)
main_df =main_df.sort_values(by=['FlightDate','OriginStateName',"airport_name"], ascending=[False, True, True])


##=== create dropdown options ====#
year_options = list(main_df['year'].unique())+["All"]
state_options = list(main_df['OriginStateName'].unique())+["All"]
airport_options = list(main_df["airport_name"].unique())+["All"]
month_options = months+["All"]
carrier_options =list(main_df['Carrier_Name'].unique())+["All"]
variable_options = explore_fields
stats_options=["mean","sum","count"]
quarter_options=list(main_df['quarter'].unique())+["All"]


## Explore the Dataset
The section below allows users to explore and compare a set of numerical attributes by appling various level of options.

In [49]:
##=== Widgets ====#
year_dropdown1 = widgets.Dropdown(options=year_options,value="All",description="Year")
carrier_dropdown1 = widgets.Dropdown(options=carrier_options,value="All",description="Carrier")
state_dropdown1 = widgets.Dropdown(options=state_options,value="All",description="State")
airport_dropdown1 = widgets.Dropdown(options=airport_options,value="All",description="Airport Name")
month_dropdown1 = widgets.Dropdown(options=month_options,value="All",description="Month")
quarter_dropdown1 = widgets.Dropdown( options=quarter_options, value="All", description="Quarter")
variables_dropdown1 = widgets.Dropdown(options=variable_options,value="flight_count",description="Attribute")
stats_dropdown1 = widgets.Dropdown(options=stats_options,value="count",description="Statistic")

## Update airport options based on selected state
def update_airport_options1(change=None):
    filtered_airports =main_df.loc[main_df["OriginStateName"] == state_dropdown1.value, "airport_name"].unique().tolist()
    airport_dropdown1.options = ["All"] + sorted(filtered_airports)
    airport_dropdown1.value = "All"

## Update month options based on selected quarter
def update_month_options1(change=None):
    quarter_month_map = {
        "Q1": ["All", "January", "February", "March"],
        "Q2": ["All", "April", "May", "June"],
        "Q3": ["All", "July", "August", "September"],
        "Q4": ["All", "October", "November", "December"]}
    if quarter_dropdown1.value == "All":
        month_dropdown1.options = month_options
    else:
        month_dropdown1.options = quarter_month_map[quarter_dropdown1.value]
    month_dropdown1.value = "All"


## Reset button set-up
reset_btn = widgets.Button(description='Reset', button_style='',  icon='refresh')
def reset_filters(b):
    year_dropdown1.value = "All"
    carrier_dropdown1.value = "All"
    state_dropdown1.value ="All"
    airport_dropdown1.value = "All"
    month_dropdown1.value = "All"
    quarter_dropdown1.value ="All"
reset_btn.on_click(reset_filters)

## main dashboard update function
output1 = widgets.Output() 
def update_dashboard1(change=None):
    with output1:
        clear_output(wait=True)
        filtered_df = main_df.copy()

        # Filter data
        if year_dropdown1.value != "All":
            filtered_df = filtered_df[filtered_df["year"] == year_dropdown1.value]
        if quarter_dropdown1.value != "All":
            filtered_df = filtered_df[filtered_df["quarter"] == quarter_dropdown1.value]
        if month_dropdown1.value != "All":
            filtered_df = filtered_df[filtered_df["month"] == month_dropdown1.value]
        if carrier_dropdown1.value != "All":
            filtered_df = filtered_df[filtered_df['Carrier_Name'] == carrier_dropdown1.value]
        if state_dropdown1.value != "All":
            filtered_df = filtered_df[filtered_df['OriginStateName'] == state_dropdown1.value]
        if airport_dropdown1.value != "All":
            filtered_df = filtered_df[filtered_df["airport_name"] == airport_dropdown1.value]
 
        if filtered_df.empty:
            print("No data found for selected filters.")
            return
        
        filtered_df= filtered_df.groupby("month", as_index=False)\
            .agg( value= (variables_dropdown1.value, stats_dropdown1.value))
        
        fig=px.bar(filtered_df, x="month",y="value",color="month",text="value",
                title=f'Summary statistics: {variables_dropdown1.value} = {stats_dropdown1.value}' )
   
        fig.update_layout(height=500,width=900,template="plotly_dark",showlegend=False)
        fig.update_traces(texttemplate='%{text:,.0f}',textposition='outside')

        display(fig)

# Widget events
year_dropdown1.observe(update_dashboard1, names="value")
quarter_dropdown1.observe(update_dashboard1, names="value")
quarter_dropdown1.observe(update_month_options1, names="value")
carrier_dropdown1.observe(update_dashboard1, names="value")
state_dropdown1.observe(update_airport_options1, names="value")
state_dropdown1.observe(update_dashboard1, names="value")
airport_dropdown1.observe(update_dashboard1, names="value")
month_dropdown1.observe(update_dashboard1, names="value")
variables_dropdown1.observe(update_dashboard1, names="value")
stats_dropdown1.observe(update_dashboard1, names="value")
# Layout
filters1 = widgets.VBox([
    widgets.HBox([year_dropdown1, quarter_dropdown1, month_dropdown1]),
    widgets.HBox([carrier_dropdown1, state_dropdown1, airport_dropdown1]),
    widgets.HBox([variables_dropdown1,stats_dropdown1]),
    reset_btn],
    layout=widgets.Layout(gap='12px',padding='10px',
        border='2px solid black',width='fit-content'))

update_airport_options1 ()
update_month_options1()
display(filters1)
display(output1)
update_dashboard1()

Output()

## What Causes the Flight Delays 
The section below allows users to explore and understand flight delays and their causes by filtering the data across different time periods and locations.

In [51]:
##=== Widgets ====#
year_dropdown2 = widgets.Dropdown(options=year_options,value="All",description="Year")
carrier_dropdown2 = widgets.Dropdown(options=carrier_options,value="All",description="Carrier")
state_dropdown2 = widgets.Dropdown(options=state_options,value="All",description="State")
airport_dropdown2 = widgets.Dropdown(options=airport_options,value="All",description="Airport Name")
month_dropdown2 = widgets.Dropdown(options=month_options,value="All",description="Month")
quarter_dropdown2 = widgets.Dropdown( options=quarter_options, value="All", description="Quarter")


## Update airport options based on selected state
def update_airport_options2(change=None):
    filtered_airports =main_df.loc[main_df["OriginStateName"] == state_dropdown2.value, "airport_name"].unique().tolist()
    airport_dropdown2.options = ["All"] + sorted(filtered_airports)
    airport_dropdown2.value = "All"
## Update month options based on selected quarter
def update_month_options2(change=None):
    quarter_month_map = {
        "Q1": ["All", "January", "February", "March"],
        "Q2": ["All", "April", "May", "June"],
        "Q3": ["All", "July", "August", "September"],
        "Q4": ["All", "October", "November", "December"]}
    if quarter_dropdown2.value == "All":
        month_dropdown2.options = month_options
    else:
        month_dropdown2.options = quarter_month_map[quarter_dropdown2.value]
    month_dropdown2.value = "All"


## Reset button set-up
reset_btn = widgets.Button(description='Reset', button_style='',  icon='refresh')
def reset_filters(b):
    year_dropdown2.value = "All"
    carrier_dropdown2.value = "All"
    state_dropdown2.value ="All"
    airport_dropdown2.value = "All"
    month_dropdown2.value = "All"
    quarter_dropdown2.value ="All"
reset_btn.on_click(reset_filters)

## main dashboard update function
output2 = widgets.Output() 
def update_dashboard2(change=None):
    with output2:
        clear_output(wait=True)
        filtered_df = main_df.copy()

        # Filter data
        if year_dropdown2.value != "All":
            filtered_df = filtered_df[filtered_df["year"] == year_dropdown2.value]
        if quarter_dropdown2.value != "All":
            filtered_df = filtered_df[filtered_df["quarter"] == quarter_dropdown2.value]
        if month_dropdown2.value != "All":
            filtered_df = filtered_df[filtered_df["month"] == month_dropdown2.value]
        if carrier_dropdown2.value != "All":
            filtered_df = filtered_df[filtered_df['Carrier_Name'] == carrier_dropdown2.value]
        if state_dropdown2.value != "All":
            filtered_df = filtered_df[filtered_df['OriginStateName'] == state_dropdown2.value]
        if airport_dropdown2.value != "All":
            filtered_df = filtered_df[filtered_df["airport_name"] == airport_dropdown2.value]
 

        if filtered_df.empty:
            print("No data found for selected filters.")
            return
        
        ## delay feilds
        delay_fields=[ 'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay','LateAircraftDelay']
        ## overall on time versus delay flights
        filtered_df['DepDel15']=filtered_df['DepDel15'].fillna(-1)
        delay_df_count=filtered_df['DepDel15'].value_counts().reset_index()
        delay_df_count['DepDel15_cat']=delay_df_count['DepDel15'].map({0:"On time", 1:"Delay",-1:"Canceled"})
        fig1=create_pie_plot(delay_df_count, 'DepDel15_cat',"count")
    
        # Delay reasons
        delay_df=filtered_df[filtered_df["DepDel15"]==1][delay_fields]
        delay_df=delay_df.melt(var_name="delay_reason", value_name="delay_time")
        delay_df=delay_df[(delay_df["delay_time"].notnull())&(delay_df["delay_time"]>0)]
        delay_df["delay_reason"]=delay_df["delay_reason"].str.replace("Delay","")
        
        delay_df_count=delay_df["delay_reason"].value_counts().reset_index()
        fig2=create_pie_plot(delay_df_count, "delay_reason","count")
    
        delay_df_count=filtered_df['CancellationCode'].value_counts().reset_index()
        fig3=create_pie_plot(delay_df_count, 'CancellationCode',"count")
  
        # delay time distribution
        fig4 =create_box_plot(delay_df, "delay_reason", "delay_time", "delay_reason")

        ## delay by months
        delay_df=filtered_df[['DepDel15',"month"]+delay_fields]
        delay_df.loc[:,delay_fields] = delay_df.loc[:, delay_fields].mask(delay_df.loc[:,delay_fields] < 15)
        delay_df=delay_df[delay_df['DepDel15']==1]
        delay_df= pd.melt(delay_df, id_vars=['DepDel15',"month"], var_name='delay_reason', value_name='delay_time')
        delay_df=delay_df[delay_df["delay_time"].notnull()]
        delay_df=delay_df[["month",'delay_reason']].value_counts().reset_index(name="delay_count")
        delay_df['delay_reason']=delay_df['delay_reason'].str.replace("Delay","")
        fig5=create_flow_plot(delay_df, "month",'delay_reason' ,"delay_count")

        ## combine plots
        combined_fig = make_subplots(rows=3,cols=2,
            specs=[[{"type": "domain"}, {"type": "domain"}],
                [{"type": "domain"},{"type": "xy"}],
                 [{"type": "domain", "colspan": 2}, None]],
      
            subplot_titles=("Flight Status","Cancellation Reasons", "Delay Reasons",
                            "Delay time distribution","Delay Reasons by Months" ),     
        )

        for trace in fig1.data:
            combined_fig.add_trace(trace, row=1, col=1)
        for trace in fig2.data:
            combined_fig.add_trace(trace, row=2, col=1)
        for trace in fig3.data:
            combined_fig.add_trace(trace, row=1, col=2)
        for trace in fig4.data:
            combined_fig.add_trace(trace, row=2, col=2)
        for trace in fig5.data:
            combined_fig.add_trace(trace, row=3, col=1)

        combined_fig.update_xaxes(title_text="Delay Reasons", row=2, col=1)
        combined_fig.update_yaxes(title_text="Delay in minutes ( log scale)",
                                    type="log", row=2, col=2,  title_font=dict(size=10))

        combined_fig.update_layout(height=900,width=900,
            template="plotly_dark",showlegend=False
        )

        display(combined_fig)

# Widget events
year_dropdown2.observe(update_dashboard2, names="value")
quarter_dropdown2.observe(update_dashboard2, names="value")
quarter_dropdown2.observe(update_month_options2, names="value")
carrier_dropdown2.observe(update_dashboard2, names="value")
state_dropdown2.observe(update_airport_options2, names="value")
state_dropdown2.observe(update_dashboard2, names="value")
airport_dropdown2.observe(update_dashboard2, names="value")
month_dropdown2.observe(update_dashboard2, names="value")

# Layout
filters2 = widgets.VBox([
    widgets.HBox([year_dropdown2, quarter_dropdown2, month_dropdown2]),
    widgets.HBox([carrier_dropdown2, state_dropdown2, airport_dropdown2]),
    reset_btn],
    layout=widgets.Layout(gap='12px',padding='10px',
        border='2px solid black',width='fit-content'))

update_airport_options2()
update_month_options2()
display(filters2)
display(output2)
update_dashboard2()

Output()

## Explore Flight patters
The section below allows users to explore flight patterns  across different time periods and locations. This [dataset](https://www.arcgis.com/home/item.html?id=cba647d88bcb4c819b01dcfba019c456) is used for US airport xy coodinates.

In [52]:


##=== Widgets ====#
year_dropdown3 = widgets.Dropdown(options=year_options,value=2025,description="Year")
state_dropdown3 = widgets.Dropdown(options=state_options,value="New York",description="State")
airport_dropdown3 = widgets.Dropdown(options=airport_options,value="JFK_New York, NY",description="Airport Name")
month_dropdown3 = widgets.Dropdown(options=month_options,value="January",description="Month")
carrier_dropdown3 = widgets.Dropdown(options=carrier_options,value="All",description="Carrier")

def update_airport_options3(change=None):
    filtered_airports =main_df.loc[main_df["OriginStateName"] == state_dropdown3.value, "airport_name"].unique().tolist()
    airport_dropdown3.options = ["All"] + sorted(filtered_airports)
    airport_dropdown3.value = "All"

output3 = widgets.Output()
def update_dashboard3(change=None):
    with output3:
        clear_output(wait=True)
        filtered_df = main_df.copy()
        #filter
        if year_dropdown3.value != "All":
             filtered_df = filtered_df[filtered_df["year"] == year_dropdown3.value]
        if carrier_dropdown3.value != "All":
            filtered_df = filtered_df[filtered_df['Carrier_Name'] == carrier_dropdown3.value]  
        if state_dropdown3.value != "All":    
            filtered_df = filtered_df[filtered_df['OriginStateName'] == state_dropdown3.value]
        if airport_dropdown3.value != "All":
            filtered_df = filtered_df[filtered_df["airport_name"] == airport_dropdown3.value]
        filtered_df = filtered_df[filtered_df["month"] == month_dropdown3.value]

        if filtered_df.empty:
            print("No data found for selected filters.")
            return   
        m = make_ipyleaflet_flight_map(filtered_df , min_count=1)
        display(m)

# Widget events
year_dropdown3.observe(update_dashboard3, names="value")
state_dropdown3.observe(update_airport_options3, names="value")
state_dropdown3.observe(update_dashboard3, names="value")
airport_dropdown3.observe(update_dashboard3, names="value")
month_dropdown3.observe(update_dashboard3, names="value")
carrier_dropdown3.observe(update_dashboard3, names="value")
# Layout
filters3 = widgets.VBox([
    widgets.HBox([year_dropdown3, carrier_dropdown3, month_dropdown3]),
    widgets.HBox([state_dropdown3, airport_dropdown3])],
    layout=widgets.Layout(gap='12px',padding='10px',
        border='2px solid black',width='fit-content'))
update_airport_options3()
display(filters3)
display(output3)
update_dashboard3()

Output()